In [1]:
# Sanity Check: Evolution Tests for CartPole, Pendulum, and Acrobot
# Using MINIMAL_SCALAR_OPS (8 fundamental operations)

import numpy as np
import time
from memory_system import MemoryConfig
from instruction_set import InstructionSet
from operation import MINIMAL_SCALAR_OPS
from population import Population, PopulationConfig
from operators import GeneticOperators
from evaluator import (
    CartPoleEvaluator, CartPoleEvaluatorConfig,
    AcrobotEvaluator, AcrobotEvaluatorConfig,
    PendulumEvaluator, PendulumEvaluatorConfig
)
from evolution_engine import EvolutionConfig, EvolutionEngine
from evolution_stats import EvolutionStatsDisplay


In [2]:
# ==================== COMMON CONFIGURATION ====================

random_seed = 42
rng = np.random.default_rng(random_seed)

# Operations (shared)
ops = MINIMAL_SCALAR_OPS
print(f"Operations: {[op().name for op in ops]}")
print(f"Total operations: {len(ops)}")

# Helper function to create memory config for each environment
def create_memory_cfg(n_obs_scalar):
    return MemoryConfig(
        n_scalar=8,
        n_vector=0,
        n_matrix=0,
        n_obs_scalar=n_obs_scalar,
        n_obs_vector=0,
        n_obs_matrix=0,
        vector_size=n_obs_scalar,
        matrix_shape=(n_obs_scalar, n_obs_scalar),
    )

# Helper to create instruction set and operators for a memory config
def create_instruction_set_and_operators(memory_cfg, rng):
    instruction_set = InstructionSet([op() for op in ops], memory_cfg)
    operators = GeneticOperators(instruction_set, rng)
    return instruction_set, operators


Operations: ['scalar_add', 'scalar_sub', 'scalar_mul', 'scalar_div_protected', 'automl_scalar_cos', 'automl_scalar_log', 'automl_scalar_exp', 'scalar_conditional']
Total operations: 8


In [3]:
# ==================== EVOLUTION PARAMETERS ====================

POPULATION_SIZE = 250
ELITISM = 100
GENERATIONS = 500
PROGRAM_LENGTH = (1, 10)
MAX_PROGRAM_LENGTH = 50

# Population config (shared across all environments)
pop_cfg = PopulationConfig(
    size=POPULATION_SIZE,
    program_length=PROGRAM_LENGTH,
    elitism=ELITISM,
    max_program_length=MAX_PROGRAM_LENGTH,
)

# Evolution config (shared across all environments)
evo_cfg = EvolutionConfig(
    max_generations=GENERATIONS,
    mutation_threshold=0.1,
    constant_mutation_rate=0.1,
    crossover_threshold=0.9,
    verbose=True,
)

print(f"Population size: {POPULATION_SIZE}")
print(f"Elitism: {ELITISM}")
print(f"Generations: {GENERATIONS}")
print(f"Program length: {PROGRAM_LENGTH}")
print(f"Max program length: {MAX_PROGRAM_LENGTH}")


Population size: 250
Elitism: 100
Generations: 500
Program length: (1, 10)
Max program length: 50


In [4]:
# ==================== TEST 1: CARTPOLE ====================
print("=" * 80)
print("TEST 1: CARTPOLE EVOLUTION")
print("=" * 80)

# CartPole-specific configuration (4 observations)
cartpole_memory_cfg = create_memory_cfg(n_obs_scalar=4)
cartpole_instruction_set, cartpole_operators = create_instruction_set_and_operators(
    cartpole_memory_cfg, np.random.default_rng(random_seed)
)

# CartPole evaluator config
cartpole_eval_cfg = CartPoleEvaluatorConfig(
    env_id="CartPole-v1",
    episodes=5,
    max_steps=500,
    output_register=7,
    render_mode="rgb_array",
    rng_seed=random_seed,
    n_jobs=None,  # Parallel
)
cartpole_eval = CartPoleEvaluator(config=cartpole_eval_cfg)

# Create population
cartpole_pop = Population(
    pop_cfg,
    cartpole_instruction_set,
    cartpole_memory_cfg,
    operators=cartpole_operators,
    rng=np.random.default_rng(random_seed),
)
cartpole_pop.initialize_random(mutate_constants=True)

# Create evolution engine
cartpole_engine = EvolutionEngine(
    population=cartpole_pop,
    operators=cartpole_operators,
    evaluator=cartpole_eval,
    config=evo_cfg,
    rng=np.random.default_rng(random_seed),
)

# Run evolution
print(f"\nStarting CartPole evolution...")
start_time = time.time()
cartpole_final = cartpole_engine.run()
cartpole_time = time.time() - start_time
print(f"\nCartPole evolution completed in {cartpole_time:.2f} seconds ({cartpole_time/60:.2f} minutes)")

# Display stats
cartpole_stats = EvolutionStatsDisplay(
    engine=cartpole_engine,
    population=cartpole_final,
    memory_cfg=cartpole_memory_cfg,
    output_registers=cartpole_eval.output_registers,
)
cartpole_stats.display(show_code=True, show_graphs=True, show_stats=True)

cartpole_eval.close()


TEST 1: CARTPOLE EVOLUTION

Starting CartPole evolution...
Evaluated 10/250 individuals
Evaluated 20/250 individuals
Evaluated 30/250 individuals
Evaluated 40/250 individuals
Evaluated 50/250 individuals
Evaluated 60/250 individuals
Evaluated 70/250 individuals
Evaluated 80/250 individuals
Evaluated 90/250 individuals
Evaluated 100/250 individuals
Evaluated 110/250 individuals
Evaluated 120/250 individuals
Evaluated 130/250 individuals
Evaluated 140/250 individuals
Evaluated 150/250 individuals
Evaluated 160/250 individuals
Evaluated 170/250 individuals
Evaluated 180/250 individuals
Evaluated 190/250 individuals
Evaluated 200/250 individuals
Evaluated 210/250 individuals
Evaluated 220/250 individuals
Evaluated 230/250 individuals
Evaluated 240/250 individuals
Evaluated 250/250 individuals
  → Initialized best_ever (fitness: 500.0000, gen: 0)
Best agent: fitness=500.0000, effective_code_rate=0.125 (1/8)

=== Generation 0 ===
Generation 0 | Population size 250
Min: 8.600, Mean: 16.528, M

Process SpawnPoolWorker-190:
Process SpawnPoolWorker-186:
Process SpawnPoolWorker-185:
Process SpawnPoolWorker-184:
Process SpawnPoolWorker-189:
Process SpawnPoolWorker-181:
Process SpawnPoolWorker-178:
Process SpawnPoolWorker-183:
Process SpawnPoolWorker-179:
Process SpawnPoolWorker-187:
Process SpawnPoolWorker-192:
Process SpawnPoolWorker-177:
Process SpawnPoolWorker-191:
Process SpawnPoolWorker-182:
Process SpawnPoolWorker-180:
Process SpawnPoolWorker-188:


KeyboardInterrupt: 

In [5]:
# ==================== TEST 2: PENDULUM ====================
print("=" * 80)
print("TEST 2: PENDULUM EVOLUTION")
print("=" * 80)

# Pendulum-specific configuration (3 observations)
pendulum_memory_cfg = create_memory_cfg(n_obs_scalar=3)
pendulum_instruction_set, pendulum_operators = create_instruction_set_and_operators(
    pendulum_memory_cfg, np.random.default_rng(random_seed)
)

# Pendulum evaluator config
pendulum_eval_cfg = PendulumEvaluatorConfig(
    env_id="Pendulum-v1",
    episodes=5,
    max_steps=500,
    output_register=7,
    render_mode="rgb_array",
    rng_seed=random_seed,
    n_jobs=None,  # Parallel
)
pendulum_eval = PendulumEvaluator(config=pendulum_eval_cfg)

# Create population
pendulum_pop = Population(
    pop_cfg,
    pendulum_instruction_set,
    pendulum_memory_cfg,
    operators=pendulum_operators,
    rng=np.random.default_rng(random_seed),
)
pendulum_pop.initialize_random(mutate_constants=True)

# Create evolution engine
pendulum_engine = EvolutionEngine(
    population=pendulum_pop,
    operators=pendulum_operators,
    evaluator=pendulum_eval,
    config=evo_cfg,
    rng=np.random.default_rng(random_seed),
)

# Run evolution
print(f"\nStarting Pendulum evolution...")
start_time = time.time()
pendulum_final = pendulum_engine.run()
pendulum_time = time.time() - start_time
print(f"\nPendulum evolution completed in {pendulum_time:.2f} seconds ({pendulum_time/60:.2f} minutes)")

# Display stats
pendulum_stats = EvolutionStatsDisplay(
    engine=pendulum_engine,
    population=pendulum_final,
    memory_cfg=pendulum_memory_cfg,
    output_registers=pendulum_eval.output_registers,
)
pendulum_stats.display(show_code=True, show_graphs=True, show_stats=True)

pendulum_eval.close()


TEST 2: PENDULUM EVOLUTION

Starting Pendulum evolution...
Evaluated 10/250 individuals
Evaluated 20/250 individuals
Evaluated 30/250 individuals
Evaluated 40/250 individuals
Evaluated 50/250 individuals
Evaluated 60/250 individuals
Evaluated 70/250 individuals
Evaluated 80/250 individuals
Evaluated 90/250 individuals
Evaluated 100/250 individuals
Evaluated 110/250 individuals
Evaluated 120/250 individuals
Evaluated 130/250 individuals
Evaluated 140/250 individuals
Evaluated 150/250 individuals
Evaluated 160/250 individuals
Evaluated 170/250 individuals
Evaluated 180/250 individuals
Evaluated 190/250 individuals
Evaluated 200/250 individuals
Evaluated 210/250 individuals
Evaluated 220/250 individuals
Evaluated 230/250 individuals
Evaluated 240/250 individuals
Evaluated 250/250 individuals
  → Initialized best_ever (fitness: -1074.3339, gen: 0)
Best agent: fitness=-1074.3339, effective_code_rate=0.200 (1/5)

=== Generation 0 ===
Generation 0 | Population size 250
Min: -1878.314, Mean: -

Process SpawnPoolWorker-2204:
Process SpawnPoolWorker-2201:
Process SpawnPoolWorker-2205:
Process SpawnPoolWorker-2200:
Process SpawnPoolWorker-2194:
Process SpawnPoolWorker-2195:
Process SpawnPoolWorker-2196:
Process SpawnPoolWorker-2208:
Process SpawnPoolWorker-2193:
Process SpawnPoolWorker-2197:
Process SpawnPoolWorker-2199:
Process SpawnPoolWorker-2207:
Process SpawnPoolWorker-2206:
Process SpawnPoolWorker-2198:
Process SpawnPoolWorker-2202:
Process SpawnPoolWorker-2203:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **

KeyboardInterrupt: 

In [6]:
# ==================== TEST 3: ACROBOT ====================
print("=" * 80)
print("TEST 3: ACROBOT EVOLUTION")
print("=" * 80)

# Acrobot-specific configuration (6 observations)
acrobot_memory_cfg = create_memory_cfg(n_obs_scalar=6)
acrobot_instruction_set, acrobot_operators = create_instruction_set_and_operators(
    acrobot_memory_cfg, np.random.default_rng(random_seed)
)

# Acrobot evaluator config
acrobot_eval_cfg = AcrobotEvaluatorConfig(
    env_id="Acrobot-v1",
    episodes=5,
    max_steps=500,
    output_register=7,
    render_mode="rgb_array",
    rng_seed=random_seed,
    n_jobs=None,  # Parallel
)
acrobot_eval = AcrobotEvaluator(config=acrobot_eval_cfg)

# Create population
acrobot_pop = Population(
    pop_cfg,
    acrobot_instruction_set,
    acrobot_memory_cfg,
    operators=acrobot_operators,
    rng=np.random.default_rng(random_seed),
)
acrobot_pop.initialize_random(mutate_constants=True)

# Create evolution engine
acrobot_engine = EvolutionEngine(
    population=acrobot_pop,
    operators=acrobot_operators,
    evaluator=acrobot_eval,
    config=evo_cfg,
    rng=np.random.default_rng(random_seed),
)

# Run evolution
print(f"\nStarting Acrobot evolution...")
start_time = time.time()
acrobot_final = acrobot_engine.run()
acrobot_time = time.time() - start_time
print(f"\nAcrobot evolution completed in {acrobot_time:.2f} seconds ({acrobot_time/60:.2f} minutes)")

# Display stats
acrobot_stats = EvolutionStatsDisplay(
    engine=acrobot_engine,
    population=acrobot_final,
    memory_cfg=acrobot_memory_cfg,
    output_registers=acrobot_eval.output_registers,
)
acrobot_stats.display(show_code=True, show_graphs=True, show_stats=True)

acrobot_eval.close()


TEST 3: ACROBOT EVOLUTION

Starting Acrobot evolution...
Evaluated 10/250 individuals
Evaluated 20/250 individuals
Evaluated 30/250 individuals
Evaluated 40/250 individuals
Evaluated 50/250 individuals
Evaluated 60/250 individuals
Evaluated 70/250 individuals
Evaluated 80/250 individuals
Evaluated 90/250 individuals
Evaluated 100/250 individuals
Evaluated 110/250 individuals
Evaluated 120/250 individuals
Evaluated 130/250 individuals
Evaluated 140/250 individuals
Evaluated 150/250 individuals
Evaluated 160/250 individuals
Evaluated 170/250 individuals
Evaluated 180/250 individuals
Evaluated 190/250 individuals
Evaluated 200/250 individuals
Evaluated 210/250 individuals
Evaluated 220/250 individuals
Evaluated 230/250 individuals
Evaluated 240/250 individuals
Evaluated 250/250 individuals
  → Initialized best_ever (fitness: -122.0000, gen: 0)
Best agent: fitness=-122.0000, effective_code_rate=0.200 (1/5)

=== Generation 0 ===
Generation 0 | Population size 250
Min: -500.000, Mean: -493.2

Process SpawnPoolWorker-2822:
Process SpawnPoolWorker-2828:
Process SpawnPoolWorker-2826:
Process SpawnPoolWorker-2823:
Process SpawnPoolWorker-2820:
Process SpawnPoolWorker-2821:
Process SpawnPoolWorker-2829:
Process SpawnPoolWorker-2825:
Process SpawnPoolWorker-2831:
Process SpawnPoolWorker-2824:
Process SpawnPoolWorker-2818:
Process SpawnPoolWorker-2819:
Process SpawnPoolWorker-2817:
Process SpawnPoolWorker-2827:
Process SpawnPoolWorker-2830:
Process SpawnPoolWorker-2832:


KeyboardInterrupt: 

In [ ]:
# ==================== SUMMARY ====================
print("=" * 80)
print("EVOLUTION SUMMARY")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Elitism: {ELITISM}")
print(f"  Generations: {GENERATIONS}")
print(f"  Operations: {len(ops)} (MINIMAL_SCALAR_OPS)")

print(f"\nResults:")
print(f"  CartPole:  Best fitness = {cartpole_final.best_ever.fitness:.2f}, Time = {cartpole_time:.2f}s")
print(f"  Pendulum:  Best fitness = {pendulum_final.best_ever.fitness:.2f}, Time = {pendulum_time:.2f}s")
print(f"  Acrobot:   Best fitness = {acrobot_final.best_ever.fitness:.2f}, Time = {acrobot_time:.2f}s")

total_time = cartpole_time + pendulum_time + acrobot_time
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print("=" * 80)
